# Rejection-sampler cost against signal-to-noise

This notebook measures how the cost of the rejection sampler grows with the
signal-to-noise of the data, for the standard RV and Gaia astrometry
parameterizations, and makes the figure for the performance appendix of the paper.

The cost of getting $X$ effective samples splits into two factors,
$$ t(X) \approx X \, \frac{J}{J_{\rm eff}} \, \frac{1}{R(N)} \quad , $$
where $R(N)$ is the number of prior samples evaluated per second, which depends on the
number of observations $N$ and the device but not on the data values, and
$J_{\rm eff}/J \approx e^{-H}$ (Case study 3), where $H$ is the information over the
sampled parameters. For informative data, each posterior width shrinks like
$1/\rho$, where $\rho$ is the total signal-to-noise of the orbit, so we expect
$J_{\rm eff}/J \propto \rho^{-d}$ with $d$ the number of sampled parameters: 4 for
the standard RV parameterization ($P$, $e$, phase, $\omega$) and 7 for the standard
Gaia parameterization ($P$, $e$, phase, $\omega$, $\Omega$, $\cos i$, and the
parallax, which is sampled because the prior on $a_0$ depends on it). If that is
right, curves for different $N$ fall on top of each other when plotted against $\rho$.

Run this on the benchmark workstation with a GPU (see `scripts/README.md`). On Linux,
the first cell pins the process to one CPU core, so the CPU throughput is for one core.
A full run takes a few hours; set `J_MAX = 10**6` for a quick check.

The results are saved to `data/benchmarks-ess.ecsv`. If that file exists, the notebook
skips the measurements and only makes the fits and the figure, so delete the file to
rerun them.

In [1]:
import os

# One CPU core for the CPU throughput numbers. This has to happen before numpy or jax
# are imported.
os.environ["XLA_FLAGS"] = (
    "--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1"
)
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[var] = "1"
if hasattr(os, "sched_setaffinity"):  # Linux only
    os.sched_setaffinity(0, {min(os.sched_getaffinity(0))})

import jax

# Enable float64 before importing harv:
jax.config.update("jax_enable_x64", True)

In [2]:
import time
import warnings
from pathlib import Path

import astropy.table as at
import harv
import harv.models as hm
import matplotlib.pyplot as plt
import numpy as np
import numpyro.distributions as dist
from harv.simulate import simulate_gaia_epoch_astrometry, simulate_rv_sb1_data
from unxt import Q

%matplotlib inline
print(f"harv {harv.__version__}, jax {jax.__version__}")
print(jax.devices())

# The sampler warns when J_eff < 3. Here that is expected for the largest rho values.
warnings.filterwarnings("ignore", message="Under-resolved rejection run")

harv 0.0.2.dev238+g4d7e5d20a, jax 0.11.2
[CpuDevice(id=0)]


In [5]:
repo_path = Path("..").resolve()
figure_path = repo_path / "paper" / "figures"
data_path = repo_path / "data"
figure_path.mkdir(exist_ok=True)
data_path.mkdir(exist_ok=True)

# The sweep and throughput results are saved here; delete the file to rerun them.
results_path = data_path / "benchmarks-ess.ecsv"
run_benchmarks = not results_path.exists()

J_MAX = 10**8  # largest prior library per source
ESS_MIN = 128  # J_eff needed to call a cell resolved
N_SEEDS = 3  # realizations of the observing times and noise per cell
RHO_VALUES = np.geomspace(1.0, 1000.0, 13)
X = 1024  # effective samples per source, as in Case study 4

N_VALUES = {"rv": (3, 6, 12), "gaia": (40, 100, 250)}
D_NL = {"rv": 4, "gaia": 7}

## Priors and simulated data

We use the prior from Case study 1 for the RVs and the prior from Case study 2 for the
Gaia astrometry, without the jitter term. The parallax prior is a truncated normal
with a width of 1 mas around the true parallax, as in Case study 2, where it is
centered on the catalog value.

Each data type has one fixed orbit. We change $\rho$ by scaling the uncertainties at
fixed $N$, and we draw new observing times and noise for each of `N_SEEDS`
realizations. RV: $P = 200$ days, $e = 0.3$, $K = 5~{\rm km\,s^{-1}}$ over 3 years.
Gaia: $P = 400$ days, $e = 0.2$, $a_0 = 0.5$ mas and $\varpi = 10$ mas over 5 years,
with random scan angles and parallax factors.

In [6]:
rv_prior = hm.StandardRV().default_prior(
    period_min=Q(2.0, "day"),
    period_max=Q(16384.0, "day"),
    sigma_K0=Q(30.0, "km/s"),
    sigma_v0=Q(150.0, "km/s"),
    eccentricity=dist.TruncatedNormal(0.5, 0.25, low=0.0, high=1.0),
)
gaia_prior = hm.StandardGaiaAstrometry().default_prior(
    period_min=Q(10.0, "day"),
    period_max=Q(2000.0, "day"),
    sigma_a0=Q(1.0, "AU"),
    sigma_pos=Q(100.0, "mas"),
    sigma_vtan=Q(100.0, "km/s"),
    parallax=harv.QD(dist.TruncatedNormal(10.0, 1.0, low=0.0), "mas"),
)
samplers = {
    "rv": harv.RejectionSampler(rv_prior, harv.RVModel()),
    "gaia": harv.RejectionSampler(
        gaia_prior,
        harv.GaiaAstrometryModel(),
        marginalized_names=("ra0", "dec0", "pmra", "pmdec", "semi_major_axis"),
    ),
}

RV_TRUTH = {
    "period": Q(200.0, "day"),
    "eccentricity": 0.3,
    "time_peri": Q(50.0, "day"),
    "arg_peri": Q(1.0, "rad"),
    "rv_semiamp": Q(5.0, "km/s"),
    "v_sys": Q(0.0, "km/s"),
    "baseline": Q(3.0, "yr"),
}
GAIA_TRUTH = {
    "period": Q(400.0, "day"),
    "eccentricity": 0.2,
    "time_peri": Q(100.0, "day"),
    "arg_peri": Q(1.0, "rad"),
    "lon_asc_node": Q(2.0, "rad"),
    "inclination": Q(1.0, "rad"),
    "semi_major_axis": Q(0.5, "mas"),
    "parallax": Q(10.0, "mas"),
    "mu_alpha": Q(5.0, "mas/yr"),
    "mu_delta": Q(-3.0, "mas/yr"),
    "baseline": Q(5.0, "yr"),
}


def simulate(kind, n_obs, err, seed):
    """Simulated data with uncertainty `err` (km/s for RV, mas for Gaia)."""
    if kind == "rv":
        data, _ = simulate_rv_sb1_data(
            seed=seed, n_obs=n_obs, rv_err=Q(err, "km/s"), **RV_TRUTH
        )
    else:
        data, _ = simulate_gaia_epoch_astrometry(
            seed=seed, n_obs=n_obs, al_error=Q(err, "mas"), **GAIA_TRUTH
        )
    return data

The total signal-to-noise ratio is the $\chi^2$ of the noiseless orbit signal after
fitting out the model without a companion (the systemic velocity for RVs; position,
proper motion and parallax for Gaia). It scales as $1/\sigma$, so we compute it once
for unit uncertainties, and then set $\sigma = \rho_1 / \rho$.

In [7]:
def rho_unit(kind, n_obs, seed):
    """Total signal-to-noise of the orbit for unit uncertainties."""
    data = simulate(kind, n_obs, 1e-30, seed)  # noiseless
    if kind == "rv":
        y = np.asarray(data.rv.to_value("km/s"))
        A = np.ones((n_obs, 1))
    else:
        y = np.asarray(data.al_position.to_value("mas"))
        psi = np.asarray(data.scan_angle.to_value("rad"))
        dt = np.asarray((data.time - data.time_ref).to_value("yr"))
        A = np.stack(
            [
                np.sin(psi),
                np.cos(psi),
                np.sin(psi) * dt,
                np.cos(psi) * dt,
                np.asarray(data.parallax_factor),
            ],
            axis=1,
        )
    resid = y - A @ np.linalg.lstsq(A, y, rcond=None)[0]
    return np.sqrt(np.sum(resid**2))

## Representative cases

These cells are run like any other and drawn as larger markers. The APOGEE
uncertainty is typical of the recalibrated visit uncertainties. The bright Gaia case
uses the median along-scan uncertainty of the Gaia-4 transits in Case study 2
(0.028 mas).

CHECK before the final run: the APOGEE value against Price-Whelan et al. (2020), and
the faint Gaia value (G of about 17) against the DR4 epoch astrometry documentation.

In [8]:
TYPICAL_ERR = {
    "rv": {"APOGEE": 0.2},  # km/s
    "gaia": {"Gaia, bright": 0.03, "Gaia, G~17": 0.4},  # mas
}

## Throughput

$R(N)$ for each data type and number of observations, on one CPU core and on the GPU
if there is one. The sampler draws the prior samples on the fly, which is how one
would run with $J = 10^8$.

In [9]:
devices = {"1 CPU core": jax.devices("cpu")[0]}
if jax.default_backend() == "gpu":
    devices["1 GPU"] = jax.devices()[0]
J_R = min(10**7, J_MAX)


def warm_time(sampler, data, J):
    """Wall time of one `run` call, after a first call that compiles."""
    key = jax.random.key(0)
    for _ in range(2):
        t0 = time.perf_counter()
        jax.block_until_ready(sampler.run(data, key=key, n_prior_samples=J, top_k=1))
    return time.perf_counter() - t0


if run_benchmarks:
    throughput = {}
    for name, device in devices.items():
        with jax.default_device(device):
            for kind, sampler in samplers.items():
                for n_obs in N_VALUES[kind]:
                    t = warm_time(sampler, simulate(kind, n_obs, 1.0, 0), J_R)
                    throughput[name, kind, n_obs] = J_R / t
                    print(f"{name}, {kind}, N = {n_obs}: {J_R / t / 1e6:.3g} M/s")

1 CPU core, rv, N = 3: 3.91 M/s
1 CPU core, rv, N = 6: 3.84 M/s
1 CPU core, rv, N = 12: 3.4 M/s
1 CPU core, gaia, N = 40: 1.03 M/s
1 CPU core, gaia, N = 100: 0.608 M/s
1 CPU core, gaia, N = 250: 0.229 M/s


## The sweep

For each cell, we run the sampler with $J = 10^6$ prior samples and multiply $J$ by 10
until $J_{\rm eff} \geq$ `ESS_MIN` or $J$ reaches `J_MAX`. Once a cell fails to
resolve, larger $\rho$ at the same $N$ and seed would fail too, so we stop there.

In [10]:
def measure(sampler, data, key):
    J = min(10**6, J_MAX)
    while True:
        s = sampler.run(data, key=key, n_prior_samples=J, top_k=1)
        ess = float(s.metadata["ln_Z_int_ess"])
        if ess >= ESS_MIN or J >= J_MAX:
            return J, ess
        J *= 10


if run_benchmarks:
    rows = []
    for kind, sampler in samplers.items():
        for n_obs in N_VALUES[kind]:
            for seed in range(N_SEEDS):
                rho1, key = rho_unit(kind, n_obs, seed), jax.random.key(seed)
                cells = [("", rho) for rho in RHO_VALUES]
                cells += [
                    (label, rho1 / err) for label, err in TYPICAL_ERR[kind].items()
                ]
                rho_fail = np.inf
                for label, rho in sorted(cells, key=lambda c: c[1]):
                    if rho >= rho_fail:  # known not to resolve
                        J, ess = J_MAX, np.nan
                    else:
                        J, ess = measure(
                            sampler, simulate(kind, n_obs, rho1 / rho, seed), key
                        )
                        if ess < ESS_MIN and not label:
                            rho_fail = rho
                    rows.append(
                        {
                            "kind": kind,
                            "N": n_obs,
                            "seed": seed,
                            "rho": rho,
                            "typical": label,
                            "J": J,
                            "ess": ess,
                        }
                    )
                print(
                    f"{kind}, N = {n_obs}, seed {seed}: unresolved above rho = {rho_fail:.3g}"
                )

    res = at.Table(rows)
    res.meta["throughput"] = [[*key, R] for key, R in throughput.items()]
    res.write(results_path)
else:
    res = at.Table.read(results_path)
    res["typical"] = res["typical"].filled("")  # ECSV reads empty strings as masked
    throughput = {tuple(key): R for *key, R in res.meta["throughput"]}

res["ratio"] = res["ess"] / res["J"]
res["resolved"] = res["ess"] >= ESS_MIN  # NaN compares False
# The devices of the run, which can differ from the devices of this machine:
device_names = list(dict.fromkeys(name for name, _, _ in throughput))

rv, N = 3, seed 0: unresolved above rho = inf
rv, N = 3, seed 1: unresolved above rho = inf
rv, N = 3, seed 2: unresolved above rho = inf
rv, N = 6, seed 0: unresolved above rho = 100
rv, N = 6, seed 1: unresolved above rho = 56.2
rv, N = 6, seed 2: unresolved above rho = 100
rv, N = 12, seed 0: unresolved above rho = 56.2
rv, N = 12, seed 1: unresolved above rho = 31.6
rv, N = 12, seed 2: unresolved above rho = 31.6


KeyboardInterrupt: 

## Results

We fit $\ln(J_{\rm eff}/J) = a + b \ln\rho$ to the resolved cells in the informative
regime ($J_{\rm eff}/J < 10^{-2}$), pooling all $N$ for each data type, and compare the
slope $b$ with $-d$. We also fit each $N$ separately to see whether the curves fall
on top of each other.

In [ ]:
fits = {}
for kind in samplers:
    for n_obs in (None, *N_VALUES[kind]):
        m = (res["kind"] == kind) & res["resolved"] & (res["ratio"] < 1e-2)
        m &= res["typical"] == ""
        if n_obs is not None:
            m &= res["N"] == n_obs
        if m.sum() < 3:
            print(f"{kind}, N = {n_obs}: too few resolved cells to fit")
            continue
        b, a = np.polyfit(np.log(res["rho"][m]), np.log(res["ratio"][m]), 1)
        fits[kind, n_obs] = (a, b, np.exp(np.mean(np.log(res["rho"][m]))))
        print(f"{kind}, N = {n_obs or 'all'}: slope {b:.2f} (predicted {-D_NL[kind]})")


def predicted_ratio(kind, rho, n_obs=None):
    """J_eff/J from the fit for this N if there is one, else the fit to all N."""
    a, b, _ = fits.get((kind, n_obs), fits[kind, None])
    return np.exp(a) * rho**b


print()
for kind in samplers:
    for label in TYPICAL_ERR[kind]:
        for n_obs in N_VALUES[kind]:
            sub = res[
                (res["kind"] == kind) & (res["typical"] == label) & (res["N"] == n_obs)
            ]
            rho = np.median(sub["rho"])
            if all(sub["resolved"]):
                ratio, how = np.median(sub["ratio"]), "measured"
            elif (kind, None) in fits:
                ratio, how = predicted_ratio(kind, rho, n_obs), "extrapolated"
            else:
                continue
            times = ", ".join(
                f"{name} {X / ratio / throughput[name, kind, n_obs]:.3g} s"
                for name in device_names
            )
            print(
                f"{label}, N = {n_obs}: rho = {rho:.3g}, J_eff/J = {ratio:.2g} "
                f"({how}); time to ESS {X}: {times}"
            )

In [ ]:
colors = ("tab:blue", "tab:orange", "tab:green")
linestyles = {"1 GPU": "-", "1 CPU core": ":"}
titles = {"rv": "Standard RV", "gaia": "Standard Gaia astrometry"}
rho_grid = np.geomspace(RHO_VALUES[0], RHO_VALUES[-1], 128)

fig, axes = plt.subplots(
    2, 2, figsize=(10, 8), sharex=True, sharey="row", layout="constrained"
)
for col, kind in enumerate(samplers):
    ax_top, ax_bot = axes[:, col]
    ax_top.set_title(titles[kind])

    for i, n_obs in enumerate(N_VALUES[kind]):
        sweep = res[
            (res["kind"] == kind) & (res["N"] == n_obs) & (res["typical"] == "")
        ]
        rhos, med = [], []
        for rho in RHO_VALUES:
            sub = sweep[sweep["rho"] == rho]
            if len(sub) and all(sub["resolved"]):
                rhos.append(rho)
                med.append(np.median(sub["ratio"]))
                ax_top.plot(
                    [rho, rho],
                    [sub["ratio"].min(), sub["ratio"].max()],
                    color=colors[i],
                    lw=1,
                )
        rhos, med = np.array(rhos), np.array(med)
        ax_top.plot(
            rhos, med, marker="o", ms=4, color=colors[i], label=f"$N = {n_obs}$"
        )

        for name, device_ls in linestyles.items():
            if name not in device_names:
                continue
            R = throughput[name, kind, n_obs]
            ax_bot.plot(rhos, X / med / R, color=colors[i], ls=device_ls)
            if (kind, None) in fits and len(rhos):
                ext = rho_grid[rho_grid >= rhos[-1]]
                ax_bot.plot(
                    ext,
                    X / predicted_ratio(kind, ext, n_obs) / R,
                    color=colors[i],
                    ls=device_ls,
                    alpha=0.4,
                )

        for label, marker in zip(TYPICAL_ERR[kind], ("*", "D"), strict=False):
            sub = res[
                (res["kind"] == kind) & (res["N"] == n_obs) & (res["typical"] == label)
            ]
            rho = np.median(sub["rho"])
            resolved = all(sub["resolved"])
            if not resolved and (kind, None) not in fits:
                continue
            if resolved:
                ratio = np.median(sub["ratio"])
            else:
                ratio = predicted_ratio(kind, rho, n_obs)
            style = {
                "marker": marker,
                "ms": 14,
                "mec": colors[i],
                "mew": 1.5,
                "ls": "none",
                "mfc": colors[i] if resolved else "none",
            }
            ax_top.plot(rho, ratio, **style)
            name = "1 GPU" if "1 GPU" in device_names else "1 CPU core"
            ax_bot.plot(rho, X / ratio / throughput[name, kind, n_obs], **style)

    if (kind, None) in fits:
        _, b, rho_mid = fits[kind, None]
        ax_top.plot(
            rho_grid,
            predicted_ratio(kind, rho_grid),
            color="0.5",
            ls="--",
            lw=1,
            zorder=-1,
            label=rf"fit: SNR$^{{{b:.1f}}}$",
        )
        # The predicted slope, through the fit at the middle of the fitted range:
        ref = predicted_ratio(kind, rho_mid) * (rho_grid / rho_mid) ** -D_NL[kind]
        ax_top.plot(
            rho_grid,
            ref,
            color="0.5",
            ls=":",
            lw=1,
            zorder=-1,
            label=rf"SNR$^{{-{D_NL[kind]}}}$",
        )
    ax_top.axhline(ESS_MIN / J_MAX, color="0.7", lw=1, zorder=-2)
    ax_top.legend(loc="lower left", fontsize=9)

    ax_top.set_ylim(1e-12, 3)
    ax_bot.set_ylim(1e-4, 1e8)
    for t, text in (
        (1, "1 s"),
        (60, "1 min"),
        (3600, "1 hr"),
        (86400, "1 day"),
        (3.15e7, "1 yr"),
    ):
        ax_bot.axhline(t, color="0.8", lw=1, zorder=-2)
        ax_bot.text(rho_grid[0], t * 1.15, text, fontsize=8, color="0.5")
    ax_bot.set_xlabel(r"total signal-to-noise ratio")

for name, ls in linestyles.items():
    if name in device_names:
        axes[1, 0].plot([], [], color="0.3", ls=ls, label=name)
axes[1, 0].legend(loc="lower right", fontsize=9)
axes[0, 0].set_ylabel(r"$J_{\rm eff} / J$")
axes[1, 0].set_ylabel(f"time to ESS = {X} [s]")
for ax in axes.flat:
    ax.set_xscale("log")
    ax.set_yscale("log")

fig.savefig(figure_path / "bench-ess.pdf")